# zenz の追加学習（コンパウンドIME 用）

かな漢字変換の小さな人工知能 **zenz-v3.2-small** に、同音異義語・話し言葉・英単語まじりの文を
追加で学ばせて、試験用の新しい模型（GGUF ファイル）を作ります。

**使い方**

1. 上のメニューの「ランタイム」→「ランタイムのタイプを変更」で、**L4 GPU** を選んで「保存」を押します
   （T4 や A100 でも動きます。設定は自動で合わせます）。
2. 下のセルを **上から順に1つずつ** 押します（セルの左の ▶ ボタン）。前のセルが終わってから次を押してください。
3. 途中で接続が切れたら、もう一度 **①から順に** 押してください。終わった所は飛ばし、学習は続きから始まります
   （途中の結果は Google ドライブの `zenz-finetune` フォルダに残ります）。
4. **このタブは最後まで開いたままにしてください。** 閉じると止まります。

かかる時間の目安（L4）: 全部で1時間半くらい。始めに見積りを表示します。

## ① 設定
ふつうは変えなくてかまいません。▶ を押してください。

In [ ]:
#@title ① 設定
RUN_NAME = "v1"            # 結果を置くフォルダの名前。別の設定で作り直すときは v2 などに変える
REPO_URL = "https://github.com/J-KAMMA/compound-ime.git"
REPO_REF = "main"          # 使うリポジトリの枝または版
DATA_SIZE = 400_000        # 学習用の問題の数
EPOCHS = 2                 # 同じ問題を何回くり返すか
LEARNING_RATE = 3e-5       # 学習の歩幅（小さめ。元の力を崩さないため）
EFFECTIVE_BATCH = 128      # 1歩で使う問題の数（GPU が違っても同じにする）
SEED = 20261015            # 乱数の種（同じなら同じ問題・同じ順番になる）
CHECKPOINT_MINUTES = 20    # 途中の結果をドライブに残す間隔（分）
EVAL_PER_KIND = 300        # 成績を比べるときの、種類ごとの問題の数

# ---- 固定した材料（版と SHA256。変えないでください）----
BASE_GGUF_URL = "https://huggingface.co/Miwa-Keita/zenz-v3.2-small-gguf/resolve/c67e03e07d215c869f591b274c1631170d3e11fe/ggml-model-Q5_K_M.gguf"
BASE_GGUF_SHA256 = "29c223d4c23327b80fd13ebb5ab2555057a46317997d5da391584ffbef0db673"
TOKENIZER_URL = "https://huggingface.co/Miwa-Keita/zenz-v3.1-small/resolve/344612e331bf3ba752a9bfef6633c93c6b933ebb/"
TOKENIZER_FILES = {
    "vocab.json": "67fd752abb091e649a9bb08bcb2b52b27cd6e3893ff5ee6e391e5555c0a90e0f",
    "merges.txt": "dde8cf4cc72f4d6eb7461a1c7dde6847916f0ad405395238aaf67501e2803ca7",
    "tokenizer_config.json": "bd7ac21f1da24ee416971702ffdd9c8489f0327914a067c65561f59c142f848d",
    "special_tokens_map.json": "f17f705b191ddb9aef0b42e40869d325d79b2a1e229e0d51d72698b5d3e4fc90",
}
WIKI_URL = "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/"
WIKI_FILES = {"20231101.ja/train-00007-of-00015.parquet": "b112e83cf73327ba2246fd9c31eb1af501e51bac8450d8420963e29132d9b04b"}
MOZC_TABLE_URL = "https://raw.githubusercontent.com/unok/mozc/2d993fd532c0df849ff3a5f6c7851b3757471ed6/src/data/preedit/romanji-hiragana.tsv"
MOZC_TABLE_SHA256 = "fcb900e311d32ced05499c86e236d1084fb0c0937398deb6311fc6c7287a7821"
LLAMA_URL = "https://github.com/fkunn1326/llama.cpp.git"
LLAMA_COMMIT = "10131b23ee0dbd9d03dfc618cec9fe3402919277"   # コンパウンドIME が動かすときと同じ版

DRIVE_ROOT = "/content/drive/MyDrive/zenz-finetune"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
WORK = "/content/work"
print("設定しました。結果の置き場所:", RUN_DIR)

## ② GPU の確認と Google ドライブへの接続
▶ を押すと、Google ドライブへの接続の許可を求める画面が出ます。**自分のアカウントを選んで許可**してください。

In [ ]:
#@title ② GPU の確認と Google ドライブへの接続
import os, subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("GPU がつながっていません。メニューの「ランタイム」→「ランタイムのタイプを変更」で "
                     "L4 GPU を選んで保存し、①からやり直してください。")
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("ドライブにつながりました:", RUN_DIR)

## ③ 道具（この学習の台本）を取ってきて、時間とユニットを見積もる

In [ ]:
#@title ③ 台本を取ってきて見積もる
import os, subprocess, sys, json
REPO = f"{WORK}/compound-ime"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
KIT = f"{REPO}/tools/zenz-finetune"
if not os.path.exists(f"{KIT}/zenz_tune.py"):
    # まだ GitHub に上がっていないときは、ドライブの zenz-finetune/kit フォルダに置いた台本を使う
    KIT = f"{DRIVE_ROOT}/kit"
    if not os.path.exists(f"{KIT}/zenz_tune.py"):
        raise SystemExit("台本が見つかりません。tools/zenz-finetune の中身（build_dataset.py と zenz_tune.py）を "
                         f"Google ドライブの {KIT} に置いてから、もう一度押してください。")
sys.path.insert(0, KIT)
import importlib, zenz_tune as z
importlib.reload(z)
print("リポジトリの版:", REPO_COMMIT, " 台本:", KIT)

PROFILE = z.gpu_profile(EFFECTIVE_BATCH)
data_built = os.path.exists(f"{RUN_DIR}/data/manifest.json")
est = z.estimate(PROFILE, DATA_SIZE, EPOCHS, data_built=data_built)
print(f"GPU の種類: {PROFILE['kind']}（{PROFILE['name']}）")
print(f"計算の精度: {PROFILE['precision']}、一度に {PROFILE['micro']} 問 × {PROFILE['accum']} 回で1歩")
print(f"見積り: 学習 約{est['train_minutes']:.0f} 分、全体 約{est['total_minutes']:.0f} 分")
if est["units"] is not None:
    print(f"コンピューティング ユニット: 約{est['units']:.1f}（1時間あたり {PROFILE['units_per_hour']}）")
print("※ 学習の速さは仮の値です。学習が始まると実際の速さで残り時間を出し直します。")

## ④ 道具を入れる
版を固定した部品を入れます（2〜3分）。

In [ ]:
#@title ④ 道具を入れる
!pip -q install "transformers==4.57.6" "sudachipy==0.6.11" "sudachidict-core==20260428" "sentencepiece==0.2.1" "protobuf>=4.21,<7"
import transformers, sudachipy, torch
print("transformers", transformers.__version__, " torch", torch.__version__, " sudachipy", sudachipy.__version__)

## ⑤ 材料を取ってくる
元の模型、分かち方の部品、ウィキペディアの一部（約220MB）、Mozc のローマ字表、llama.cpp を取ってきて、
すべて SHA256（ファイルの指紋）を確かめます（3〜5分）。

In [ ]:
#@title ⑤ 材料を取ってくる
import os, subprocess
BASE_GGUF = z.download(BASE_GGUF_URL, f"{DRIVE_ROOT}/cache/zenz-v3.2-small-Q5_K_M.gguf", BASE_GGUF_SHA256)
TOKENIZER_DIR = f"{DRIVE_ROOT}/cache/tokenizer"
for name, digest in TOKENIZER_FILES.items():
    z.download(TOKENIZER_URL + name, f"{TOKENIZER_DIR}/{name}", digest)
ROMAJI_TABLE = z.download(MOZC_TABLE_URL, f"{DRIVE_ROOT}/cache/romanji-hiragana.tsv", MOZC_TABLE_SHA256)
WIKI_PATHS = []
if not os.path.exists(f"{RUN_DIR}/data/manifest.json"):
    for name, digest in WIKI_FILES.items():
        WIKI_PATHS.append(str(z.download(WIKI_URL + name, f"{WORK}/wiki/{os.path.basename(name)}", digest)))
LLAMA = f"{WORK}/llama.cpp"
if not os.path.isdir(f"{LLAMA}/.git"):
    os.makedirs(LLAMA, exist_ok=True)
    subprocess.run(["git", "init", "-q", LLAMA], check=True)
    subprocess.run(["git", "-C", LLAMA, "remote", "add", "origin", LLAMA_URL], check=True)
subprocess.run(["git", "-C", LLAMA, "fetch", "-q", "--depth", "1", "origin", LLAMA_COMMIT], check=True)
subprocess.run(["git", "-C", LLAMA, "checkout", "-q", "FETCH_HEAD"], check=True)
print("そろいました。llama.cpp の版:",
      subprocess.run(["git", "-C", LLAMA, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

## ⑥ 元の模型を学習できる形に戻す
zenz-v3.2-small は量子化（圧縮）された GGUF でしか配られていないため、圧縮をほどいて学習できる形にします（1分）。

In [ ]:
#@title ⑥ 元の模型を学習できる形に戻す
import os, json
BASE_DIR = f"{RUN_DIR}/base"
if not os.path.exists(f"{BASE_DIR}/from_gguf.json"):
    info = z.gguf_to_hf(BASE_GGUF, TOKENIZER_DIR, BASE_DIR, LLAMA)
else:
    info = json.load(open(f"{BASE_DIR}/from_gguf.json"))
print("戻しました:", info["config"], "埋め込みの出どころ:", info["embedding_from"])

## ⑦ 練習問題を作る
ウィキペディアの文に読みを付け、同音異義語・英単語まじり・話し言葉の問題を作ります。
**試験の文（tests/conversion/）と同じ文・それを含む文は自動で捨てます。**（10〜20分。GPU は使いません）

In [ ]:
#@title ⑦ 練習問題を作る
import os, json, shlex
DATA_DIR = f"{RUN_DIR}/data"
if not os.path.exists(f"{DATA_DIR}/manifest.json"):
    tmp = f"{WORK}/data"
    if not WIKI_PATHS:
        raise SystemExit("ウィキペディアのファイルがありません。⑤をもう一度押してください。")
    WIKI_ARGS = " ".join(shlex.quote(p) for p in WIKI_PATHS)
    !python "{KIT}/build_dataset.py" --repo "{REPO}" --out "{tmp}" --romaji-table "{ROMAJI_TABLE}" --wiki {WIKI_ARGS} --size {DATA_SIZE} --seed {SEED} --progress 5000
    if not os.path.exists(f"{tmp}/manifest.json"):
        raise SystemExit("問題作りが失敗しました。上の表示を確かめてください。")
    !mkdir -p "{DATA_DIR}" && cp "{tmp}"/* "{DATA_DIR}/"
manifest = json.load(open(f"{DATA_DIR}/manifest.json"))
print("学習用:", manifest["counts"]["train_by_kind"])
print("確かめ用:", manifest["counts"]["valid_by_kind"])
print("試験と重なって捨てた文:", manifest["test_exclusion"]["excluded_sentences"])

## ⑧ 元の模型を確かめる
文の終わりの記号を元の模型に合わせ、学習前の成績を測っておきます（5分）。

In [ ]:
#@title ⑧ 元の模型を確かめる
import os, json, numpy as np, torch
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer
DEVICE = torch.device("cuda")
tok = AutoTokenizer.from_pretrained(BASE_DIR)
slow = GPT2Tokenizer.from_pretrained(BASE_DIR)
sample = [json.loads(l) for _, l in zip(range(300), open(f"{DATA_DIR}/valid.jsonl", encoding="utf-8"))]
for r in sample:
    text = r["prompt"] + r["output"]
    assert tok(text, add_special_tokens=False)["input_ids"] == slow(text, add_special_tokens=False)["input_ids"], text
print("分かち方の部品: 速い版と遅い版の結果が同じです")

base = GPT2LMHeadModel.from_pretrained(BASE_DIR).to(DEVICE).eval()
END_FILE = f"{RUN_DIR}/end_token.json"
if not os.path.exists(END_FILE):
    probe, _ = z.load_examples(f"{DATA_DIR}/valid.jsonl", tok, end_id=3)
    choice = z.choose_end_token(base, probe, DEVICE, PROFILE["precision"])
    json.dump(choice, open(END_FILE, "w"))
END_ID = json.load(open(END_FILE))["end_id"]
print("文の終わりの記号:", END_ID, tok.convert_ids_to_tokens(END_ID), json.load(open(END_FILE))["votes"])

def cached(name):
    path = f"{RUN_DIR}/{name}-{END_ID}.npz"
    if not os.path.exists(path):
        ex, dropped = z.load_examples(f"{DATA_DIR}/{name}.jsonl", tok, END_ID)
        ex.save(path)
        print(f"{name}: {len(ex)} 問（長すぎて外した {dropped} 問）")
    return z.Examples.load(path)
TRAIN = cached("train"); VALID = cached("valid")
avg = float(TRAIN.length.mean())
est = z.estimate(PROFILE, len(TRAIN), EPOCHS, avg_tokens=avg, data_built=True)
print(f"1問の平均 {avg:.0f} 字。学習の見積り 約{est['train_minutes']:.0f} 分")

BASE_EVAL = f"{RUN_DIR}/eval_base.json"
if not os.path.exists(BASE_EVAL):
    report = z.evaluate(base, VALID, tok, END_ID, DEVICE, PROFILE["precision"], per_kind=EVAL_PER_KIND, seed=SEED)
    json.dump(report, open(BASE_EVAL, "w"), ensure_ascii=False, indent=1)
z.print_report(json.load(open(BASE_EVAL)))
for kind, r in json.load(open(BASE_EVAL)).items():
    for e in r["examples"][:1]:
        print(kind, "| 正解:", e["answer"], "| 元の模型:", e["got"])
del base; torch.cuda.empty_cache()

## ⑨ 学習する（いちばん長い）
進み具合と残り時間・残りのユニットを表示します。**途中で切れても、①から押し直せば続きから始まります。**

In [ ]:
#@title ⑨ 学習する
import os
FINAL_DIR = f"{RUN_DIR}/final"
if os.path.exists(f"{FINAL_DIR}/train_state.json"):
    print("学習はもう終わっています:", FINAL_DIR)
else:
    z.train(BASE_DIR, TRAIN, VALID, f"{RUN_DIR}/checkpoints", FINAL_DIR, epochs=EPOCHS, lr=LEARNING_RATE,
            profile=PROFILE, seed=SEED, checkpoint_minutes=CHECKPOINT_MINUTES)

## ⑩ 学習の前と後の成績を比べる
同じ確かめ用の問題（学習には使っていない）で比べます。「差」がマイナスの種類があれば、表示の注意を読んでください。

In [ ]:
#@title ⑩ 成績を比べる
import json, torch
from transformers import GPT2LMHeadModel
TUNED_EVAL = f"{RUN_DIR}/eval_tuned.json"
if not os.path.exists(TUNED_EVAL):
    tuned = GPT2LMHeadModel.from_pretrained(FINAL_DIR).to(DEVICE).eval()
    report = z.evaluate(tuned, VALID, tok, END_ID, DEVICE, PROFILE["precision"], per_kind=EVAL_PER_KIND, seed=SEED)
    json.dump(report, open(TUNED_EVAL, "w"), ensure_ascii=False, indent=1)
    del tuned; torch.cuda.empty_cache()
before, after = json.load(open(BASE_EVAL)), json.load(open(TUNED_EVAL))
z.print_report(before, after)
worse = [k for k in ("wiki_whole", "wiki_context", "wiki_span") if k in after and after[k]["exact"] < before[k]["exact"] - 0.01]
if worse:
    print("\n注意: ふつうの文の成績が下がっています:", worse,
          "\nこの模型は試験で悪くなる見込みが高いです。LEARNING_RATE を 1.5e-5 に下げるか EPOCHS を 1 にし、"
          "RUN_NAME を変えて作り直すことを考えてください。")
else:
    print("\nふつうの文の成績は下がっていません。")

## ⑪ GGUF にして縮める
llama.cpp（コンパウンドIME と同じ版）で GGUF にし、元と同じ Q5_K_M に量子化します。
比べるための「学習していない元の模型を同じ手順で作り直したもの」（対照）も作ります（10分）。

In [ ]:
#@title ⑪ GGUF にして縮める
import os, subprocess, sys
BIN = f"{LLAMA}/build/bin"
if not os.path.exists(f"{BIN}/llama-quantize"):
    subprocess.run(["cmake", "-S", LLAMA, "-B", f"{LLAMA}/build", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF",
                    "-DCMAKE_BUILD_TYPE=Release"], check=True, capture_output=True)
    subprocess.run(["cmake", "--build", f"{LLAMA}/build", "-j", str(os.cpu_count() or 2), "--target",
                    "llama-quantize", "llama-tokenize", "llama-simple"], check=True, capture_output=True)
OUT = f"{RUN_DIR}/result"
os.makedirs(OUT, exist_ok=True)
NAME = f"zenz-v3.2-small-ft-{RUN_NAME}"
TARGETS = {"tuned": (FINAL_DIR, f"{OUT}/{NAME}-Q5_K_M.gguf"),
           "control": (BASE_DIR, f"{OUT}/zenz-v3.2-small-roundtrip-Q5_K_M.gguf")}
for label, (src, dst) in TARGETS.items():
    if os.path.exists(dst):
        continue
    z.prepare_for_convert(src)
    f32 = f"{WORK}/{label}-f32.gguf"
    subprocess.run([sys.executable, f"{LLAMA}/convert_hf_to_gguf.py", src, "--outtype", "f32", "--outfile", f32,
                    "--model-name", NAME if label == "tuned" else "zenz-v3.2-small-roundtrip"], check=True)
    subprocess.run([f"{BIN}/llama-quantize", f32, dst, "Q5_K_M"], check=True, capture_output=True)
    print("作りました:", dst)

## ⑫ できた GGUF を確かめる
元の GGUF と設定・重みの形・量子化の種類が同じか、分かち方が同じか、llama.cpp で動かした答えが学習した模型の答えと合うかを調べます。

In [ ]:
#@title ⑫ できた GGUF を確かめる
import json, torch
from transformers import GPT2LMHeadModel
original = z.gguf_summary(BASE_GGUF, LLAMA)
ok = True
for label, (_, path) in TARGETS.items():
    problems = z.compare_summaries(original, z.gguf_summary(path, LLAMA))
    print(f"{label}: 元の GGUF との違い {len(problems)} 件")
    for p in problems[:10]:
        print("   ", p)
    ok &= not problems
records = [json.loads(l) for _, l in zip(range(40), open(f"{DATA_DIR}/valid.jsonl", encoding="utf-8"))]
same_tok = 0
for r in records[:20]:
    text = r["prompt"] + r["output"]
    same_tok += z.llama_tokenize_ids(f"{BIN}/llama-tokenize", TARGETS["tuned"][1], text) == \
        tok(text, add_special_tokens=False)["input_ids"]
print(f"分かち方が同じ: {same_tok}/20")
ok &= same_tok == 20
tuned = GPT2LMHeadModel.from_pretrained(FINAL_DIR).to(DEVICE).eval()
prompts = [tok(r["prompt"], add_special_tokens=False)["input_ids"] for r in records]
hf = [tok.decode(x) for x in z.greedy(tuned, prompts, {END_ID}, DEVICE, "fp32")]
agree = 0
for r, h in zip(records, hf):
    g = z.llama_simple_answer(f"{BIN}/llama-simple", TARGETS["tuned"][1], r["prompt"])
    agree += g == h
print(f"llama.cpp（量子化後）と学習した模型の答えが同じ: {agree}/{len(records)}（量子化のため全部は合わないことがあります）")
del tuned; torch.cuda.empty_cache()
print("確認: よし" if ok and agree >= len(records) * 0.8 else "確認: 気になる点があります。上の表示を見てください。")

## ⑬ 結果をドライブに保存する
SHA256（指紋）、説明書き（出典と利用条件）、問題の目録、成績を `result` フォルダにまとめます。

In [ ]:
#@title ⑬ 結果をまとめる
import json, shutil
lines = []
for label, (_, path) in TARGETS.items():
    digest = z.sha256_file(path)
    lines.append(f"{digest}  {os.path.basename(path)}")
    print(label, os.path.basename(path), os.path.getsize(path) // 1_000_000, "MB", digest)
open(f"{OUT}/SHA256SUMS.txt", "w").write("\n".join(lines) + "\n")
tuned_path = TARGETS["tuned"][1]
open(f"{OUT}/MODEL_CARD.md", "w").write(z.MODEL_CARD.format(name=os.path.basename(tuned_path), base_url=BASE_GGUF_URL,
    epochs=EPOCHS, lr=LEARNING_RATE, sha256=z.sha256_file(tuned_path)))
shutil.copy(f"{DATA_DIR}/manifest.json", f"{OUT}/manifest.json")
json.dump({"base": json.load(open(BASE_EVAL)), "tuned": json.load(open(TUNED_EVAL)), "repo_commit": REPO_COMMIT,
           "llama_commit": LLAMA_COMMIT, "end_id": END_ID, "profile": PROFILE},
          open(f"{OUT}/eval.json", "w"), ensure_ascii=False, indent=1)
print("\nできあがり。Google ドライブの次の場所にあります:", OUT)
print("※ この模型はまだ公開しないでください。比べ方は README.md を見てください。")